In [1]:
from sqlalchemy.orm.attributes import get_state_history
%load_ext autoreload
%autoreload 2
"""1. Run pipeline to record checkpoints into SQLite database."""

from dotenv import load_dotenv
load_dotenv(override=True)

from data.sqlite_db import get_sqlite_checkpointer
from graphs.code_review_parentgraph import build_code_review_graph

# Build graph with SqliteSaver:
graph = build_code_review_graph(checkpointer=get_sqlite_checkpointer())

# Code for review:
code_snippet = """
def is_positive(num: int) -> bool:
    if num > 0:
        return True
    return False
"""

config = {"configurable": {"thread_id": "cli_review_session_05"}}

print("Running pipeline to generate checkpoints...")
result = graph.invoke(
    {
        "code": code_snippet,
        "language": "python",
        "file_path": None,
    },
    config=config,
)
print("Pipeline run finished! Checkpoints recorded into SQLite.")

Running pipeline to generate checkpoints...


D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(


Pipeline run finished! Checkpoints recorded into SQLite.


In [2]:
"""2. Inspect chronological checkpoint history recorded in SQLite database."""

from data.sqlite_db import get_sqlite_checkpointer
from graphs.code_review_parentgraph import build_code_review_graph
from utils.time_travel import inspect_state_history

graph = build_code_review_graph(checkpointer=get_sqlite_checkpointer())

# Retrieve full state timeline for this thread:
history = inspect_state_history(graph, thread_id="cli_review_session_05")

print(f"Total Checkpoints in DB: {len(history)}\n")

for item in history:
    step_num = item["step"]
    node_name = item["node"]
    next_nodes = item["next_nodes"]
    chk_id = item["checkpoint_id"]
    keys = item["state_keys"]

    print(f"• Step {step_num}: Node [{node_name}] -> Next: {next_nodes}")
    print(f"  Checkpoint ID: {chk_id}")
    print(f"  State Keys: {keys}\n")

Deserializing unregistered type output_schemas.code_review.ReviewResult from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('output_schemas.code_review', 'ReviewResult')]
Deserializing unregistered type output_schemas.code_review.TestSuite from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('output_schemas.code_review', 'TestSuite')]
Deserializing unregistered type output_schemas.code_review.TestExecutionResult from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('output_schemas.code_review', 'TestExecutionResult')]


Total Checkpoints in DB: 4

• Step -1: Node [input] -> Next: ['__start__']
  Checkpoint ID: 1f1a6fc3-7945-644c-bfff-ebd4e9b84b2c
  State Keys: []

• Step 0: Node [loop] -> Next: ['reviewer']
  Checkpoint ID: 1f1a6fc3-7954-66cf-8000-26a08a3856a5
  State Keys: ['code', 'language', 'file_path']

• Step 1: Node [loop] -> Next: ['test_generator']
  Checkpoint ID: 1f1a6fc3-9a3a-6039-8001-85b72b46712e
  State Keys: ['code', 'language', 'file_path', 'review', 'error']

• Step 2: Node [loop] -> Next: ['refactor_gate']
  Checkpoint ID: 1f1a6fc3-b440-682a-8002-5adefd249add
  State Keys: ['code', 'language', 'file_path', 'review', 'tests', 'test_execution', 'error']



In [3]:
for snapshot in graph.get_state_history(config, subgraphs=True):
    print(snapshot.metadata["checkpoint_ns"], snapshot.next)

TypeError: Pregel.get_state_history() got an unexpected keyword argument 'subgraphs'

In [4]:
for snapshot in graph.get_state_history(config, subgraphs=True):
  print(snapshot.metadata["checkpoint_ns"], snapshot.next)

TypeError: Pregel.get_state_history() got an unexpected keyword argument 'subgraphs'